# 04 - Async basics, with fake API calls

**Goal:** understand how the gateway can run many LLM calls at once, and how it stays
under free-tier limits while doing so. Everything here uses *fake* calls (`asyncio.sleep`),
so it costs no API quota and each cell finishes in a few seconds.

**How to use it:** run the cells top to bottom, one at a time. Read the output, then read
the note under it. Predict the timing *before* you run each cell.

**One notebook-specific thing:** Jupyter already has an event loop running, so you can
write `await something()` directly in a cell. In a normal `.py` script you would instead
write `asyncio.run(main())`. (That is what `scripts/smoke_groq.py` does.)

In [1]:
import asyncio
import logging
import random
import time

# Show the retry layer's log messages in the notebook output (force=True replaces any
# handler the notebook already installed, so messages appear exactly once)
logging.basicConfig(level=logging.WARNING, format="    log: %(message)s", force=True)

# A tiny stopwatch so every print can show "how many seconds since the cell started"
_t0 = time.perf_counter()

def reset_clock() -> None:
    """Restart the stopwatch (call at the top of each timing cell)."""
    global _t0
    _t0 = time.perf_counter()

def log(message: str) -> None:
    """Print a message prefixed with the seconds elapsed since reset_clock()."""
    print(f"[{time.perf_counter() - _t0:5.2f}s] {message}")

async def fake_llm_call(name: str, seconds: float = 1.0) -> str:
    """Pretend to be an LLM call: wait (network + model time), then return a result."""
    # `await` means "I'm just waiting; let other tasks run in the meantime"
    await asyncio.sleep(seconds)
    return f"{name} done"

## Part 1 - What `async def` and `await` actually do

An `async def` function is a **coroutine function**. Calling it does *not* run it.
It gives you a coroutine object: a paused job that something has to `await`.

In [2]:
coro = fake_llm_call("example")

# Calling the function ran nothing. We just got a paused coroutine object.
print(type(coro))

# Close it so Python doesn't warn that it was "never awaited"
# (forgetting `await` is the #1 async bug; that warning is how you notice it)
coro.close()

# Awaiting it is what actually runs it and gives back the return value
print(await fake_llm_call("example"))

<class 'coroutine'>
example done


### Sequential vs concurrent

Same five calls, each taking 1 second. **Predict the total time for each version.**

In [3]:
# Sequential: each call finishes before the next one starts
reset_clock()
results = []
for i in range(5):
    results.append(await fake_llm_call(f"call-{i}"))
log("sequential finished")

[ 5.00s] sequential finished


In [4]:
# Concurrent: start all five, then wait for all of them together
reset_clock()
results = await asyncio.gather(*(fake_llm_call(f"call-{i}") for i in range(5)))
log("concurrent finished")
print(results)

[ 1.01s] concurrent finished
['call-0 done', 'call-1 done', 'call-2 done', 'call-3 done', 'call-4 done']


**Why that works:** there is still only **one thread**. While call 0 is waiting on the
network, Python switches to call 1, then call 2, and so on, and comes back when a wait is
over. The switching happens only at `await` points. This is perfect for LLM calls, because
almost all of their time is *waiting*, not computing.

**Ordering:** `gather` returns results in the order you passed them in, not in the order
they finished. Run this and watch the log lines versus the final list:

In [5]:
async def timed_call(name: str, seconds: float) -> str:
    log(f"{name} started")
    await asyncio.sleep(seconds)
    log(f"{name} finished")
    return name.upper()

reset_clock()
results = await asyncio.gather(
    timed_call("slow", 3),
    timed_call("fast", 1),
    timed_call("medium", 2),
)
# Finish order was fast, medium, slow, but results are in INPUT order
print(results)

[ 0.00s] slow started
[ 0.00s] fast started
[ 0.00s] medium started
[ 1.00s] fast finished
[ 2.00s] medium finished
[ 3.00s] slow finished
['SLOW', 'FAST', 'MEDIUM']


## Part 2 - The big pitfall: blocking calls

`await asyncio.sleep(...)` politely hands control back. A normal blocking call, like
`time.sleep`, the `requests` library, or a *synchronous* SDK client, does not. While it
runs, **nothing else in the program can run**, so concurrency silently disappears.

In [6]:
async def blocking_call(i: int) -> int:
    time.sleep(0.5)            # BLOCKS the whole event loop for 0.5s
    return i

async def friendly_call(i: int) -> int:
    await asyncio.sleep(0.5)   # lets other tasks run while waiting
    return i

reset_clock()
await asyncio.gather(*(blocking_call(i) for i in range(4)))
log("4 blocking calls finished")        # predict: ~2.0s (they ran one at a time)

reset_clock()
await asyncio.gather(*(friendly_call(i) for i in range(4)))
log("4 friendly calls finished")        # predict: ~0.5s (they overlapped)

[ 2.01s] 4 blocking calls finished
[ 0.50s] 4 friendly calls finished


**Why the gateway uses `AsyncOpenAI` and `client.aio.models.generate_content`:** the
*synchronous* clients you used in notebooks 01 to 03 would block exactly like
`time.sleep` does here. The async clients `await` the network instead.

## Part 3 - When one call fails inside `gather`

For a batch endpoint this matters a lot: if 1 of 100 requests fails, you still want the
other 99 results.

In [7]:
async def maybe_fail(name: str) -> str:
    # "bad" fails quickly (0.1s); everything else needs 0.5s
    await asyncio.sleep(0.1 if name == "bad" else 0.5)
    if name == "bad":
        raise ValueError("the 'bad' call failed")
    log(f"{name} finished")
    return name

# DEFAULT behavior: gather raises the first error immediately
reset_clock()
try:
    await asyncio.gather(*(maybe_fail(n) for n in ["a", "bad", "c"]))
except ValueError as exc:
    log(f"gather raised: {exc}")

# Notice what happens next: the other calls were NOT cancelled. They keep running in the
# background, and their results are thrown away. (Wait a moment to see them finish.)
await asyncio.sleep(1)

[ 0.10s] gather raised: the 'bad' call failed
[ 0.50s] a finished
[ 0.50s] c finished


In [8]:
# return_exceptions=True: failures come back as VALUES in the results list
reset_clock()
results = await asyncio.gather(
    *(maybe_fail(n) for n in ["a", "bad", "c"]),
    return_exceptions=True,
)

for result in results:
    if isinstance(result, Exception):
        print("ERROR:", type(result).__name__, result)
    else:
        print("ok:   ", result)

[ 0.50s] a finished
[ 0.50s] c finished
ok:    a
ERROR: ValueError the 'bad' call failed
ok:    c


**Takeaway for `client.batch()`:** use `return_exceptions=True` (or catch errors inside each
task) so one bad request can't throw away the other results. Python 3.11+ also has
`asyncio.TaskGroup`, which instead *cancels everything* on the first failure. That is
the opposite trade-off, and not what we want for independent requests.

## Part 4 - Timeouts

A call that hangs forever would hold a slot forever. `asyncio.timeout` puts a ceiling on
any block of `await`ed work.

In [9]:
async def hangs() -> None:
    await asyncio.sleep(10)   # pretend the provider never answers

reset_clock()
try:
    async with asyncio.timeout(1.0):   # give up after 1 second
        await hangs()
except TimeoutError:
    log("gave up waiting")

[ 1.01s] gave up waiting


(Our provider adapters already set a 30s timeout inside the SDK clients. `asyncio.timeout`
is the tool for a deadline around a *whole* operation, such as a retried call or a batch.)

## Part 5 - Capping concurrency with a semaphore

Firing 100 requests at once would trip free-tier rate limits immediately. An
`asyncio.Semaphore(3)` is a bouncer: at most 3 tasks may be inside the `async with` block;
the rest wait their turn.

In [12]:
stats = {"in_flight": 0, "peak": 0}

async def limited_call(i: int, sem: asyncio.Semaphore) -> int:
    async with sem:                       # waits here if 3 calls are already running
        stats["in_flight"] += 1
        stats["peak"] = max(stats["peak"], stats["in_flight"])
        log(f"call {i} running ({stats['in_flight']} in flight)")
        await asyncio.sleep(1)            # the "work"
        stats["in_flight"] -= 1
    return i

reset_clock()
sem = asyncio.Semaphore(10)
await asyncio.gather(*(limited_call(i, sem) for i in range(10)))
log(f"done. peak concurrency was {stats['peak']}")
# 10 calls, 3 at a time, 1s each -> 4 waves -> about 4 seconds

[ 0.00s] call 0 running (1 in flight)
[ 0.00s] call 1 running (2 in flight)
[ 0.00s] call 2 running (3 in flight)
[ 0.00s] call 3 running (4 in flight)
[ 0.00s] call 4 running (5 in flight)
[ 0.00s] call 5 running (6 in flight)
[ 0.00s] call 6 running (7 in flight)
[ 0.00s] call 7 running (8 in flight)
[ 0.00s] call 8 running (9 in flight)
[ 0.00s] call 9 running (10 in flight)
[ 1.00s] done. peak concurrency was 10


**Try it:** change `Semaphore(3)` to `Semaphore(1)` (fully sequential) and `Semaphore(10)`
(everything at once), and watch the total time.

## Part 6 - A semaphore limits *simultaneous* calls, not calls *per minute*

Free tiers limit **requests per minute** and **tokens per minute**. Three quick calls in a
row can finish in one second and still blow a per-minute budget. For that we need a rate
limiter: a gate that spaces requests out over time.

This one hands out time slots. Each call reserves the next free slot, and the cost says how
long that slot is. A cost of 1 per request gives "requests per second"; a cost of
`tokens` gives "tokens per second".

In [13]:
class RateLimiter:
    """Spaces calls out so the average rate never exceeds `units_per_second`."""

    def __init__(self, units_per_second: float) -> None:
        self._rate = units_per_second
        # The earliest moment the NEXT caller may start
        self._next_free = 0.0
        # Makes "read next_free, then update it" atomic between concurrent tasks
        self._lock = asyncio.Lock()

    async def acquire(self, cost: float = 1.0) -> None:
        """Wait until it is this caller's turn. `cost` is requests or tokens."""
        async with self._lock:
            now = time.monotonic()
            start = max(now, self._next_free)
            # Reserve this caller's slot: the next caller must wait cost/rate seconds
            self._next_free = start + cost / self._rate
        # Sleep OUTSIDE the lock so other callers can reserve their own slots meanwhile
        await asyncio.sleep(max(0.0, start - now))

In [14]:
limiter = RateLimiter(units_per_second=5)   # 5 requests per second -> one every 0.2s

async def rate_limited_call(i: int) -> int:
    await limiter.acquire()
    log(f"request {i} sent")
    await asyncio.sleep(0.5)                # the "work"
    return i

reset_clock()
await asyncio.gather(*(rate_limited_call(i) for i in range(8)))
# All 8 tasks were started at once, but the requests leave 0.2s apart

[ 0.03s] request 0 sent
[ 0.20s] request 1 sent
[ 0.40s] request 2 sent
[ 0.60s] request 3 sent
[ 0.80s] request 4 sent
[ 1.00s] request 5 sent
[ 1.20s] request 6 sent
[ 1.40s] request 7 sent


[0, 1, 2, 3, 4, 5, 6, 7]

### Tokens per minute: the math from earlier, now as code

Groq's free-tier sample cap was 8,000 tokens per minute, and your structured-output call
used about 547 tokens. Passing `cost=547` to the limiter turns that cap into a spacing:

In [15]:
tokens_per_minute_cap = 8_000
tokens_per_request = 547

print(f"max requests per minute: about {tokens_per_minute_cap / tokens_per_request:.1f}")

token_limiter = RateLimiter(units_per_second=tokens_per_minute_cap / 60)
spacing = tokens_per_request / (tokens_per_minute_cap / 60)
print(f"so requests must be spaced about {spacing:.1f}s apart")

# In the real gateway: await token_limiter.acquire(cost=estimated_tokens) before each call.
# The cost has to be an ESTIMATE, because we only learn the true usage after the response.

max requests per minute: about 14.6
so requests must be spaced about 4.1s apart


## Part 7 - Putting it together: a mini batch runner

This combines everything, and uses your **real** retry layer from `src/llm_gateway/retry.py`:

- a **semaphore** caps simultaneous calls
- a **rate limiter** spaces requests out, and *every retry attempt* goes through it too
- **`with_retries`** retries temporary failures (`RetryableError`) but not permanent ones
- **`return_exceptions=True`** keeps one failure from losing the other results

The fake provider fails randomly 40% of the time with a retryable error, and always
rejects one input with a non-retryable error.

In [16]:
from llm_gateway.errors import InvalidRequestError, RetryableError
from llm_gateway.retry import RetryPolicy, with_retries

random.seed(7)                       # same "random" failures every run, so runs are comparable
attempts_by_item: dict[str, int] = {}

async def fake_provider_call(item: str) -> str:
    """One attempt against a fake provider."""
    attempts_by_item[item] = attempts_by_item.get(item, 0) + 1
    await asyncio.sleep(0.3)
    if item == "bad-input":
        # A permanent failure: the retry layer must NOT retry this
        raise InvalidRequestError("provider rejected this input", provider="fake")
    if random.random() < 0.4:
        # A temporary failure: the retry layer SHOULD retry this
        raise RetryableError("simulated 503", provider="fake")
    return f"result for {item}"

async def run_batch(
    items: list[str],
    *,
    concurrency: int,
    limiter: RateLimiter,
    policy: RetryPolicy,
) -> list[str | BaseException]:
    """Run every item concurrently, within the concurrency and rate limits."""
    sem = asyncio.Semaphore(concurrency)

    async def run_one(item: str) -> str:
        async with sem:                           # at most `concurrency` items at a time

            async def one_attempt() -> str:
                # Every attempt, including retries, waits for a rate-limit slot first
                await limiter.acquire()
                return await fake_provider_call(item)

            return await with_retries(one_attempt, policy)

    # return_exceptions=True: failed items come back as exception objects, not a crash
    return await asyncio.gather(*(run_one(i) for i in items), return_exceptions=True)

In [25]:
items = [f"doc-{i}" for i in range(8)] + ["bad-input"]
attempts_by_item.clear()

reset_clock()
results = await run_batch(
    items,
    concurrency=3,
    limiter=RateLimiter(units_per_second=10),
    policy=RetryPolicy(max_attempts=4, base_delay=0.1, max_delay=1.0),
)
log("batch finished")

print()
for item, result in zip(items, results, strict=True):
    status = f"ERROR ({type(result).__name__})" if isinstance(result, Exception) else "ok"
    print(f"{item:10} {status:28} attempts={attempts_by_item[item]}")

    log: fake: attempt 1/4 failed (RetryableError), retrying in 0.1s
    log: fake: attempt 1/4 failed (RetryableError), retrying in 0.1s
    log: fake: attempt 1/4 failed (RetryableError), retrying in 0.1s
    log: fake: attempt 2/4 failed (RetryableError), retrying in 0.2s


[ 1.76s] batch finished

doc-0      ok                           attempts=1
doc-1      ok                           attempts=1
doc-2      ok                           attempts=1
doc-3      ok                           attempts=1
doc-4      ok                           attempts=3
doc-5      ok                           attempts=2
doc-6      ok                           attempts=2
doc-7      ok                           attempts=1
bad-input  ERROR (InvalidRequestError)  attempts=1


**What to notice in the output:**
- The `log:` lines are your retry layer announcing each retry and its wait.
- Some items took 2 or 3 attempts but still succeeded.
- An unlucky item can use all 4 attempts and still end as a `RetryableError` (it was
  `doc-4` in my run). The retry layer gave up and re-raised the last error, as designed.
- `bad-input` shows `attempts=1`: a permanent error is never retried.
- Results line up with `items` by position, so a caller can tell which request failed.

## Part 8 - Experiments (do these, they are the learning)

1. Change `concurrency=3` to `1`, then to `20`. How does total time change? Where does the
   rate limiter become the bottleneck instead?
2. Change the limiter to `units_per_second=2`. Predict the total time before running.
3. Change `random.random() < 0.4` to `< 1.0` so every call fails. What happens to each
   item, and how many attempts does each get?
4. Remove `return_exceptions=True` from `run_batch`. What happens to the results now?
5. Move `await limiter.acquire()` *outside* `one_attempt` (so retries skip the limiter).
   Why would that be a bug against a real provider?

## How this becomes `client.py`

| Notebook piece | In the gateway |
|---|---|
| `Semaphore(concurrency)` | `LLM_MAX_CONCURRENCY` from `.env` |
| `RateLimiter` | one per provider (requests per minute, and tokens per minute for Groq) |
| `with_retries(one_attempt, policy)` | `RetryingProvider` around each adapter |
| `gather(..., return_exceptions=True)` | `batch()` returning one result or error per request |
| fake call | `provider.generate(request)` |